# 08 - Explainable AI: SHAP, Anchors, Counterfactuals, and Stakeholder Interpretation

This notebook explains the selected credit-risk model from Notebook 06/07. It is designed for a professional banking/finance portfolio: model interpretation, classification-metric communication, Deepchecks diagnostics, SHAP global/local explanations, anchor-style rules, counterfactual scenario diagnostics, and regulator-ready summaries.

**Important governance framing**

The explanations in this notebook are for model-risk review, portfolio monitoring, and analyst interpretation. Counterfactuals are diagnostic model-sensitivity scenarios, not promises of approval and not customer instructions.

## 1. Notebook responsibility

Notebook 08 covers Part 2: Explainable AI.

It includes:

- Classification metrics and confusion-matrix interpretation.
- Recall, precision, F1, false positives, and false negatives.
- Stakeholder and business-cost implications.
- Deepchecks model diagnostics, with fallback diagnostics when Deepchecks is unavailable.
- SHAP global feature importance.
- SHAP summary and dependence plots.
- SHAP local explanations for individual predictions.
- Anchor-style intuitive explanations.
- Counterfactual scenario diagnostics.
- Ethical, regulatory, and stakeholder communication notes.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
HTML_DIR = PROJECT_ROOT / "reports" / "html"
for folder in [TABLE_DIR, FIGURE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

In [2]:
from credit_risk.explainability.anchors_analysis import build_anchor_like_rules
from credit_risk.explainability.counterfactuals import best_counterfactual_per_account, generate_counterfactual_scenarios
from credit_risk.explainability.deepchecks_analysis import (
    fallback_model_weakness_diagnostics,
    model_enhancement_recommendations,
    run_optional_deepchecks_model_evaluation,
)
from credit_risk.explainability.shap_analysis import (
    business_regulator_summary_model_insights,
    build_explainability_readiness_gate,
    classification_metrics_at_threshold,
    compute_tree_shap_values,
    confusion_matrix_summary,
    feature_columns_from_metadata,
    get_model_split,
    individual_top_contributions,
    load_explainability_artifacts,
    plot_global_shap_bar,
    plot_numeric_shap_dependence,
    plot_shap_beeswarm,
    predict_scores,
    select_explanation_sample,
    stakeholder_metric_impact_summary,
    summarize_global_shap,
    summarize_grouped_shap,
    summarize_probability_deciles,
    summarize_shap_by_segment,
)

## 2. Load champion model, threshold, metadata, and modelling data

This notebook uses the operational champion carried forward from Notebook 06/07. If both a ranking champion and operational champion exist, this notebook explains the operational champion because it is the model-threshold pair intended for business use.

In [3]:
artifacts = load_explainability_artifacts(project_root=PROJECT_ROOT)
feature_cols = feature_columns_from_metadata(artifacts.metadata, artifacts.modeling_df)

print("Champion model:", artifacts.champion_model_name)
print("Model artifact:", artifacts.model_path)
print("Metadata artifact:", artifacts.metadata_path)
print("Threshold artifact:", artifacts.threshold_path)
print("Operating threshold:", artifacts.recommended_threshold)
print("Feature count:", len(feature_cols))

Champion model: xgboost_weighted_baseline
Model artifact: D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\model_artifacts\champion_model.joblib
Metadata artifact: D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\model_artifacts\model_feature_metadata.joblib
Threshold artifact: D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\tables\recommended_threshold_summary.csv
Operating threshold: 0.56
Feature count: 40


## 3. Overall model performance analysis

The confusion matrix and classification metrics are interpreted in business language. In credit risk, recall and false negatives often matter more than raw accuracy because missed defaulters can create credit losses. Precision and false positives matter because they drive review workload and customer friction.

In [4]:
X_train, y_train, _ = get_model_split(artifacts.modeling_df, artifacts.metadata, split_name="train")
X_test, y_test, identity_test = get_model_split(artifacts.modeling_df, artifacts.metadata, split_name="test")

scores = pd.Series(
    predict_scores(artifacts.champion_model, X_test),
    index=X_test.index,
    name="predicted_default_probability",
)

metrics = classification_metrics_at_threshold(
    y_test,
    scores,
    threshold=artifacts.recommended_threshold,
    false_negative_cost=5000,
    false_positive_cost=500,
)
metrics_df = pd.DataFrame([{"model_name": artifacts.champion_model_name, "dataset": "test", **metrics}])
metrics_df.to_csv(TABLE_DIR / "08_xai_classification_metrics_at_operating_threshold.csv", index=False)
metrics_df

,model_name,dataset,threshold,accuracy,balanced_accuracy,precision,recall,f1,mcc,roc_auc,pr_auc,brier_score,review_rate,business_cost,true_negative,false_positive,false_negative,true_positive,default_count,non_default_count
0,xgboost_weighted_baseline,test,0.56,0.727422,0.679973,0.190877,0.622052,0.292117,0.226423,0.747839,0.214653,0.201375,0.294649,5848500.0,13533,4807,689,1134,1823,18340


In [5]:
confusion_df = confusion_matrix_summary(metrics)
confusion_df.to_csv(TABLE_DIR / "08_xai_confusion_matrix_interpretation.csv", index=False)
confusion_df

,cell,count,business_meaning,stakeholder_impact
0,True Negative,13533,Account predicted lower risk and did not default.,No manual review needed; operational workload ...
1,False Positive,4807,Account predicted high risk but did not default.,Creates review workload and possible customer ...
2,False Negative,689,Account predicted lower risk but later defaulted.,Potential credit loss because the account was ...
3,True Positive,1134,Account predicted high risk and later defaulted.,Useful early-warning identification for credit...


In [6]:
stakeholder_df = stakeholder_metric_impact_summary(metrics)
stakeholder_df.to_csv(TABLE_DIR / "08_xai_stakeholder_metric_impact_summary.csv", index=False)
stakeholder_df

,metric,value,interpretation,stakeholder_impact
0,Accuracy,7.274215e-01,Share of all accounts classified correctly at ...,Can be misleading when defaults are rare; shou...
1,Recall,6.220516e-01,Share of actual defaulters captured by the hig...,Higher recall reduces missed default risk but ...
2,Precision,1.908770e-01,Share of reviewed/high-risk accounts that actu...,Higher precision improves reviewer efficiency ...
3,F1 Score,2.921175e-01,Harmonic mean of precision and recall.,"Useful single-number trade-off metric, but bus..."
4,False Negatives,6.890000e+02,Defaults missed by the model threshold.,Usually more expensive in credit risk because ...
5,False Positives,4.807000e+03,Non-defaulters flagged for review.,Creates operational cost and can affect custom...
6,Review Rate,2.946486e-01,Share of accounts routed to manual/high-risk r...,Connects model output to staffing capacity and...
7,Business Cost,5.848500e+06,Illustrative cost using false-negative and fal...,Supports threshold choice and model governance...


## 4. Understanding score bands and model separation

Score deciles help business users see whether higher model scores correspond to higher observed default rates. This is useful for communicating whether the model is ranking risk directionally correctly.

In [7]:
decile_df = summarize_probability_deciles(y_test, scores.to_numpy(), artifacts.recommended_threshold)
decile_df.to_csv(TABLE_DIR / "08_xai_probability_decile_profile.csv", index=False)
decile_df

,score_decile,row_count,average_score,min_score,max_score,observed_default_rate,review_rate_at_threshold
0,"(0.00254, 0.109]",2017,0.071571,0.003540,0.109293,0.012890,0.000000
1,"(0.109, 0.179]",2016,0.143175,0.109347,0.179347,0.018353,0.000000
2,"(0.179, 0.259]",2016,0.217334,0.179402,0.258804,0.032738,0.000000
3,"(0.259, 0.339]",2016,0.299596,0.258818,0.338855,0.036706,0.000000
4,"(0.339, 0.414]",2017,0.377112,0.338862,0.414184,0.054041,0.000000
5,"(0.414, 0.484]",2016,0.450333,0.414245,0.484142,0.070933,0.000000
6,"(0.484, 0.556]",2016,0.519369,0.484148,0.556050,0.109127,0.000000
7,"(0.556, 0.63]",2016,0.592133,0.556060,0.629990,0.136409,0.946429
8,"(0.63, 0.705]",2016,0.667083,0.630018,0.705428,0.185020,1.000000
9,"(0.705, 0.893]",2017,0.764104,0.705503,0.893485,0.247893,1.000000


## 5. Deepchecks for model enhancement

This section attempts to run Deepchecks. If Deepchecks is not installed, the notebook still produces fallback diagnostics for weak segments, false positives, false negatives, and model-improvement recommendations.

In [8]:
segment_columns = [
    col for col in [
        "loan_category", "employment_type", "home", "loan_to_income_band",
        "interest_rate_band", "amount_band", "income_band", "tenure_band",
    ] if col in X_test.columns
]

segment_columns

['loan_category',
 'employment_type',
 'home',
 'loan_to_income_band',
 'interest_rate_band',
 'income_band',
 'tenure_band']

In [9]:
deepchecks_status = run_optional_deepchecks_model_evaluation(
    model=artifacts.champion_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    output_html_path=HTML_DIR / "08_deepchecks_model_evaluation.html",
)
deepchecks_status.to_csv(TABLE_DIR / "08_deepchecks_execution_status.csv", index=False)
deepchecks_status

deepchecks - WARNING - Cannot use model's built-in feature importance on a Scikit-learn Pipeline, using permutation feature importance calculation instead
deepchecks - INFO - Calculating permutation feature importance. Expected to finish in 106 seconds


,check_name,status,finding,recommended_action
0,Deepchecks model_evaluation suite,completed,Deepchecks report saved to D:\Banking and Fina...,"Review failed/warned checks for weak segments,..."


In [10]:
weakness_df = fallback_model_weakness_diagnostics(
    X=X_test,
    y=y_test,
    scores=scores,
    threshold=artifacts.recommended_threshold,
    segment_columns=segment_columns,
)
weakness_df.to_csv(TABLE_DIR / "08_deepchecks_fallback_model_weakness_diagnostics.csv", index=False)
weakness_df.head(20)

,diagnostic_area,segment,row_count,observed_default_rate,average_score,review_rate,false_positive_rate_within_segment,false_negative_rate_within_segment,recommended_action
8,segment_weakness::employment_type,Salaried,6663,0.091100,0.407313,0.286958,0.232028,0.036170,"Investigate segment-specific calibration, feat..."
9,segment_weakness::employment_type,Self-Employed,1422,0.149086,0.560494,0.577356,0.463432,0.035162,"Investigate segment-specific calibration, feat..."
10,segment_weakness::employment_type,Unknown,12078,0.083126,0.394065,0.265607,0.215433,0.032952,"Investigate segment-specific calibration, feat..."
11,segment_weakness::home,Mortgage,9906,0.077832,0.377851,0.238542,0.196144,0.035433,"Investigate segment-specific calibration, feat..."
12,segment_weakness::home,Own,1849,0.093023,0.400678,0.280151,0.220660,0.033532,"Investigate segment-specific calibration, feat..."
13,segment_weakness::home,Rent,8398,0.104311,0.450273,0.363658,0.292093,0.032746,"Investigate segment-specific calibration, feat..."
25,segment_weakness::income_band,100K-150K,2389,0.063625,0.316587,0.150272,0.124320,0.037673,"Investigate segment-specific calibration, feat..."
27,segment_weakness::income_band,75K-100K,3605,0.074064,0.360960,0.225243,0.185853,0.034674,"Investigate segment-specific calibration, feat..."
28,segment_weakness::income_band,<=50K,7129,0.111096,0.474449,0.388273,0.310703,0.033525,"Investigate segment-specific calibration, feat..."
26,segment_weakness::income_band,50K-75K,6183,0.092027,0.420454,0.308265,0.249717,0.033479,"Investigate segment-specific calibration, feat..."


In [11]:
recommendations_df = model_enhancement_recommendations(metrics, weakness_df)
recommendations_df.to_csv(TABLE_DIR / "08_model_enhancement_recommendations.csv", index=False)
recommendations_df

,area,finding,recommendation,expected_effect
0,False negatives,False negatives at threshold: 689,Review missed-default cohorts and consider seg...,"Reduce missed defaults, possibly at higher rev..."
1,False positives,False positives at threshold: 4807,Review high-score non-default cohorts; improve...,Improve precision and reduce reviewer workload.
2,Calibration,Brier score: 0.2014,Consider calibration testing after champion se...,More reliable risk probabilities and threshold...
3,Data and features,"Performance ceiling appears feature-limited, n...","Prioritize leakage-safe repayment trend, utili...",Potentially higher PR-AUC and better separatio...
4,Weak segment review,Highest FN-rate segment in fallback diagnostic...,Inspect SHAP drivers and data quality for this...,Improved governance understanding of model wea...


## 6. Exploring SHAP

SHAP values explain how features push the model score upward or downward. Global SHAP summarizes important drivers across many accounts. Local SHAP explains an individual prediction.

For runtime control, change `MAX_SHAP_ROWS`. A sample of 1,000 to 1,500 rows is usually enough for portfolio reporting.

In [12]:
MAX_SHAP_ROWS = 1500
sample_idx = select_explanation_sample(
    X_test,
    y_test,
    scores.to_numpy(),
    threshold=artifacts.recommended_threshold,
    max_rows=MAX_SHAP_ROWS,
    high_risk_rows=250,
    near_threshold_rows=250,
    random_state=42,
)
X_sample = X_test.loc[sample_idx].copy()
print("SHAP sample rows:", len(X_sample))

SHAP sample rows: 1500


In [13]:
shap_df, X_transformed = compute_tree_shap_values(artifacts.champion_model, X_sample)
raw_feature_names = feature_cols
print("SHAP matrix shape:", shap_df.shape)
shap_df.head()

SHAP matrix shape: (1500, 40)


,numeric__amount,numeric__interest_rate,numeric__tenure_years,numeric__total_income_pa,numeric__dependents,numeric__amount_missing_raw_flag,numeric__amount_non_positive_flag,numeric__industry_placeholder_zero_flag,numeric__work_experience_placeholder_zero_flag,numeric__amount_missing_flag,numeric__employment_type_missing_flag,numeric__tier_of_employment_missing_flag,numeric__industry_missing_flag,numeric__work_experience_missing_flag,numeric__married_missing_flag,numeric__social_profile_missing_flag,numeric__is_verified_missing_flag,numeric__loan_to_income_ratio,numeric__core_data_quality_issue_count,numeric__has_core_data_quality_issue,numeric__broad_data_quality_issue_count,numeric__has_broad_data_quality_issue,numeric__income_to_loan_buffer,numeric__loan_to_income_missing_flag,numeric__high_interest_flag,numeric__long_tenure_flag,numeric__high_loan_to_income_flag,numeric__borrower_missingness_flag_count,numeric__has_any_missingness_flag,loan_category,employment_type,tier_of_employment,work_experience,home,is_verified,loan_amount_band,income_band,interest_rate_band,tenure_band,loan_to_income_band
55092,0.053210,0.940616,-0.011548,-0.073246,0.028387,0.518924,0.0,0.009117,0.004110,0.373995,0.002942,-0.000368,-0.004642,-0.000901,0.021482,0.006595,-0.000512,-0.005081,-0.319534,-0.069335,-0.115851,-0.007428,0.023246,0.090424,0.073248,0.001472,0.001793,0.008570,0.000211,0.443239,0.015694,-0.006315,-0.004755,0.035238,-0.001967,0.058682,-0.007523,0.035245,0.000032,-0.003167
36305,0.061202,0.882954,-0.024513,-0.059215,0.061834,0.531693,0.0,-0.008918,-0.000641,0.366000,-0.003607,0.000107,0.015775,0.002679,-0.010932,0.006332,0.001673,0.008615,-0.317350,-0.072471,-0.042127,-0.008469,0.034191,0.093594,0.145956,-0.001192,0.005768,-0.026069,0.000197,0.404094,0.006771,0.003189,0.017058,-0.050328,-0.023349,0.066266,-0.003044,0.009189,-0.001379,-0.001856
39808,0.047333,0.961163,-0.033581,0.133388,0.025662,0.587859,0.0,0.005838,0.002058,0.410768,0.001820,0.000105,-0.005560,-0.001981,0.017384,-0.006730,-0.000444,-0.010475,-0.310921,-0.068132,-0.110514,-0.008646,0.042414,0.093359,0.077811,-0.000260,0.001483,0.047471,0.000197,0.014216,-0.016558,-0.020133,-0.009473,0.048611,0.021190,0.039783,0.010408,0.047159,-0.001379,-0.003014
46539,0.056797,0.780753,-0.032758,-0.076084,0.031786,0.554634,0.0,0.009187,0.004110,0.395352,0.000715,-0.000193,-0.005033,-0.000901,0.020067,0.005442,-0.000347,-0.018542,-0.322576,-0.069038,-0.110525,-0.007705,0.034449,0.091348,0.134568,-0.004719,0.005457,0.003969,0.000300,0.461731,0.010130,0.009857,-0.004856,0.022724,-0.022728,0.058784,-0.001806,0.008477,-0.001379,-0.002605
46549,0.058264,0.476036,-0.030929,0.155479,-0.022546,0.593479,0.0,0.004807,0.002058,0.462994,0.001319,0.000185,-0.005344,-0.001188,0.015330,0.006292,0.000137,-0.023908,-0.299397,-0.068433,-0.106756,-0.008304,0.037953,0.100390,0.081046,-0.002896,0.005420,0.080853,0.000197,0.387818,-0.015524,-0.019634,-0.005964,0.023472,0.003717,0.079018,0.005590,0.041265,-0.001379,-0.003071


## 7. SHAP summary plot and global feature importance

The tables below show both transformed-feature importance and grouped business-feature importance. Grouped importance is usually better for stakeholder communication because one-hot encoded levels are recombined to their original feature.

In [14]:
global_importance = summarize_global_shap(shap_df, raw_feature_names=raw_feature_names)
grouped_importance = summarize_grouped_shap(global_importance)

global_importance.to_csv(TABLE_DIR / "08_xai_global_shap_importance_transformed.csv", index=False)
grouped_importance.to_csv(TABLE_DIR / "08_xai_grouped_shap_importance.csv", index=False)

grouped_importance.head(20)

,raw_feature,feature_label,mean_abs_shap,mean_shap,transformed_feature_count,positive_contribution_share
0,interest_rate,Interest Rate,0.628578,0.001553,1,0.562667
1,amount_missing_flag,Amount Missing Flag,0.192372,0.060985,1,0.298000
2,amount_missing_raw_flag,Amount Missing Raw Flag,0.181842,0.058320,1,0.298000
3,broad_data_quality_issue_count,Broad Data Quality Issue Count,0.158858,-0.032899,1,0.156000
4,total_income_pa,Total Income Pa,0.141970,-0.020161,1,0.520667
5,core_data_quality_issue_count,Core Data Quality Issue Count,0.106240,-0.077969,1,0.689333
6,amount,Amount,0.063021,-0.021421,1,0.525333
7,income_to_loan_buffer,Income To Loan Buffer,0.062619,0.040654,1,0.799333
8,high_interest_flag,High Interest Flag,0.061995,0.011855,1,0.322000
9,tenure_years,Tenure Years,0.061812,-0.011569,1,0.636667


In [15]:
plot_global_shap_bar(global_importance, FIGURE_DIR / "08_xai_global_transformed_shap_top_features.png", top_n=20)
plot_global_shap_bar(grouped_importance.rename(columns={"raw_feature": "transformed_feature"}), FIGURE_DIR / "08_xai_global_grouped_shap_top_features.png", top_n=20)

common_idx = shap_df.index.intersection(X_transformed.index)
plot_shap_beeswarm(shap_df.loc[common_idx], X_transformed.loc[common_idx], FIGURE_DIR / "08_xai_shap_summary_beeswarm.png", max_display=20)

print("Saved SHAP summary figures to reports/figures/.")

Saved SHAP summary figures to reports/figures/.


## 8. Business/regulator summary model insights

This section translates model drivers into business meaning and governance review notes.

In [16]:
regulator_summary = business_regulator_summary_model_insights(grouped_importance, top_n=15)
regulator_summary.to_csv(TABLE_DIR / "08_xai_business_regulator_summary_model_insights.csv", index=False)
regulator_summary

,raw_feature,feature_label,mean_abs_shap,directional_note,business_interpretation,regulatory_governance_note
0,interest_rate,Interest Rate,0.628578,Positive mean SHAP increases model score,Pricing/risk-based rate signal. Review for cor...,"Document lineage, timing, proxy-risk assessmen..."
1,amount_missing_flag,Amount Missing Flag,0.192372,Positive mean SHAP increases model score,Model uses this feature as a predictive signal...,"Document lineage, timing, proxy-risk assessmen..."
2,amount_missing_raw_flag,Amount Missing Raw Flag,0.181842,Positive mean SHAP increases model score,Model uses this feature as a predictive signal...,"Document lineage, timing, proxy-risk assessmen..."
3,broad_data_quality_issue_count,Broad Data Quality Issue Count,0.158858,Negative mean SHAP lowers model score on average,Model uses this feature as a predictive signal...,"Document lineage, timing, proxy-risk assessmen..."
4,total_income_pa,Total Income Pa,0.141970,Negative mean SHAP lowers model score on average,Capacity-to-pay signal. Use carefully; ensure ...,"Document lineage, timing, proxy-risk assessmen..."
5,core_data_quality_issue_count,Core Data Quality Issue Count,0.106240,Negative mean SHAP lowers model score on average,Model uses this feature as a predictive signal...,"Document lineage, timing, proxy-risk assessmen..."
6,amount,Amount,0.063021,Negative mean SHAP lowers model score on average,Exposure size signal. Higher exposure can incr...,"Document lineage, timing, proxy-risk assessmen..."
7,income_to_loan_buffer,Income To Loan Buffer,0.062619,Positive mean SHAP increases model score,Model uses this feature as a predictive signal...,"Document lineage, timing, proxy-risk assessmen..."
8,high_interest_flag,High Interest Flag,0.061995,Positive mean SHAP increases model score,Model uses this feature as a predictive signal...,"Document lineage, timing, proxy-risk assessmen..."
9,tenure_years,Tenure Years,0.061812,Negative mean SHAP lowers model score on average,Model uses this feature as a predictive signal...,"Document lineage, timing, proxy-risk assessmen..."


## 9. SHAP dependence plots and subsegment insights

Dependence plots show how a numeric feature contributes across its range. Subsegment SHAP summaries help compare model drivers across portfolio groups.

In [17]:
for raw_feature in ["interest_rate", "loan_to_income_ratio", "total_income_pa", "amount", "delinq_2yrs", "number_of_loans"]:
    plot_numeric_shap_dependence(
        shap_df=shap_df,
        X_raw=X_test,
        raw_feature=raw_feature,
        output_path=FIGURE_DIR / f"08_xai_shap_dependence_{raw_feature}.png",
        raw_feature_names=raw_feature_names,
    )
print("Saved dependence plots where source features were available.")

Saved dependence plots where source features were available.


In [18]:
subsegment_shap = summarize_shap_by_segment(
    X_raw=X_test.loc[shap_df.index],
    shap_df=shap_df,
    grouped_importance=grouped_importance,
    segment_columns=segment_columns,
    raw_feature_names=raw_feature_names,
    top_n_features=8,
    min_segment_rows=100,
)
subsegment_shap.to_csv(TABLE_DIR / "08_xai_shap_feature_dependence_subsegment_insights.csv", index=False)
subsegment_shap.head(30)

,segment_column,segment_value,segment_row_count,raw_feature,feature_label,mean_abs_shap,mean_shap,positive_contribution_share
16,employment_type,Salaried,489,interest_rate,Interest Rate,0.663051,-0.042928,0.541922
17,employment_type,Salaried,489,amount_missing_flag,Amount Missing Flag,0.190574,0.055533,0.288344
18,employment_type,Salaried,489,amount_missing_raw_flag,Amount Missing Raw Flag,0.177048,0.053945,0.288344
19,employment_type,Salaried,489,broad_data_quality_issue_count,Broad Data Quality Issue Count,0.153428,-0.041083,0.141104
20,employment_type,Salaried,489,total_income_pa,Total Income Pa,0.140817,-0.023221,0.543967
21,employment_type,Salaried,489,core_data_quality_issue_count,Core Data Quality Issue Count,0.100853,-0.070088,0.705521
23,employment_type,Salaried,489,income_to_loan_buffer,Income To Loan Buffer,0.064960,0.038244,0.766871
22,employment_type,Salaried,489,amount,Amount,0.061034,-0.013907,0.552147
24,employment_type,Self-Employed,128,interest_rate,Interest Rate,0.599908,0.239839,0.671875
25,employment_type,Self-Employed,128,amount_missing_flag,Amount Missing Flag,0.277956,0.224666,0.593750


## 10. Individual prediction explanations with SHAP

Local explanations are generated for high-risk accounts and near-threshold accounts. This supports analyst review and model-governance documentation.

In [19]:
high_risk_idx = scores.sort_values(ascending=False).head(20).index.tolist()
near_threshold_idx = (scores - artifacts.recommended_threshold).abs().sort_values().head(20).index.tolist()
local_candidate_idx = pd.Index(high_risk_idx + near_threshold_idx).drop_duplicates().tolist()

missing_local = [idx for idx in local_candidate_idx if idx not in shap_df.index]
if missing_local:
    local_shap, _ = compute_tree_shap_values(artifacts.champion_model, X_test.loc[missing_local])
    shap_df = pd.concat([shap_df, local_shap], axis=0)

individual_df = individual_top_contributions(
    X_raw=X_test,
    identity=identity_test,
    shap_df=shap_df,
    scores=scores,
    candidate_indices=local_candidate_idx,
    threshold=artifacts.recommended_threshold,
    raw_feature_names=raw_feature_names,
    top_n=6,
)
individual_df.to_csv(TABLE_DIR / "08_xai_individual_prediction_shap_explanations.csv", index=False)
individual_df.head(20)

,user_id,record_sequence,defaulter,split,row_index,predicted_default_probability,operating_threshold,predicted_high_risk,top_positive_drivers,top_negative_drivers
0,2498303,1,1,test,55092,0.893485,0.56,1,Interest Rate (+0.9406); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3195); Broad...
1,3082919,1,1,test,36305,0.889163,0.56,1,Interest Rate (+0.8830); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3174); Has C...
2,2985700,1,1,test,39808,0.885159,0.56,1,Interest Rate (+0.9612); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3109); Broad...
3,2775991,1,1,test,46539,0.884053,0.56,1,Interest Rate (+0.7808); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3226); Broad...
4,2776421,1,0,test,46549,0.882921,0.56,1,Amount Missing Raw Flag (+0.5935); Interest Ra...,Core Data Quality Issue Count (-0.2994); Broad...
5,3018328,1,0,test,38600,0.879084,0.56,1,Interest Rate (+0.9896); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3113); Broad...
6,3046302,1,0,test,37358,0.878947,0.56,1,Interest Rate (+1.0183); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3279); Broad...
7,6612416,1,0,test,3395,0.878249,0.56,1,Interest Rate (+0.9153); Broad Data Quality Is...,Amount (-0.0758); Amount Missing Raw Flag (-0....
8,2941704,1,1,test,40982,0.876165,0.56,1,Interest Rate (+0.9877); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3099); Broad...
9,2532982,1,1,test,54110,0.874884,0.56,1,Interest Rate (+0.6075); Amount Missing Raw Fl...,Core Data Quality Issue Count (-0.3153); Broad...


## 11. Anchors: intuitive explanations

Anchor-style explanations translate complex model behaviour into simple conditions. These are not policy rules. They are explanation aids showing when similar accounts are often treated as high risk by the model.

In [20]:
anchor_rules = build_anchor_like_rules(
    X_reference=X_test,
    y_reference=y_test,
    scores_reference=scores,
    X_explain=X_test,
    shap_explain=shap_df,
    candidate_indices=high_risk_idx[:15],
    threshold=artifacts.recommended_threshold,
    raw_feature_names=raw_feature_names,
    max_conditions=3,
)
anchor_rules.to_csv(TABLE_DIR / "08_xai_anchor_style_rules.csv", index=False)
anchor_rules.head(15)

,row_index,predicted_default_probability,operating_threshold,predicted_high_risk,rule,condition_count,explanation_type,interpretation,governance_note,coverage,covered_rows,model_high_risk_precision,observed_default_rate
0,55092,0.893485,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.003720,75,1.000000,0.293333
1,36305,0.889163,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.003720,75,1.000000,0.293333
2,46539,0.884053,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.003720,75,1.000000,0.293333
3,54110,0.874884,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.003720,75,1.000000,0.293333
4,3395,0.878249,0.56,1,Interest Rate >= 11.84 AND Broad Data Quality ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.002133,43,0.953488,0.302326
5,39808,0.885159,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.106234,2142,0.915033,0.212418
6,46549,0.882921,0.56,1,Amount Missing Raw Flag = 1 AND Interest Rate ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.106234,2142,0.915033,0.212418
7,38600,0.879084,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.106234,2142,0.915033,0.212418
8,37358,0.878947,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.106234,2142,0.915033,0.212418
9,40982,0.876165,0.56,1,Interest Rate >= 11.84 AND Amount Missing Raw ...,3,anchor_style_shap_rule,"When these conditions hold, the model often tr...","Use as a model explanation aid, not as a stand...",0.106234,2142,0.915033,0.212418


## 12. Counterfactuals for business insight

Counterfactual scenarios show how model probability changes when selected features are modified. These are diagnostic sensitivity tests, not customer instructions or guaranteed outcomes.

In [21]:
cf_candidate_idx = pd.Index(high_risk_idx[:5] + near_threshold_idx[:10]).drop_duplicates().tolist()
counterfactuals = generate_counterfactual_scenarios(
    pipeline=artifacts.champion_model,
    X_candidates=X_test.loc[cf_candidate_idx],
    baseline_scores=scores,
    threshold=artifacts.recommended_threshold,
    reference_df=X_train,
)
best_cfs = best_counterfactual_per_account(counterfactuals)

counterfactuals.to_csv(TABLE_DIR / "08_xai_counterfactual_scenarios.csv", index=False)
best_cfs.to_csv(TABLE_DIR / "08_xai_best_counterfactual_per_account.csv", index=False)
best_cfs.head(20)

,row_index,scenario,action_type,changed_features,baseline_probability,scenario_probability,probability_change,absolute_probability_reduction,operating_threshold,baseline_above_threshold,scenario_below_threshold,crosses_below_threshold,governance_note
0,71,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.560203,0.415989,-0.144215,0.144215,0.56,True,True,True,Diagnostic model-sensitivity scenario; not a c...
1,29557,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.559893,0.188273,-0.371620,0.371620,0.56,False,True,False,Diagnostic model-sensitivity scenario; not a c...
2,35922,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.560121,0.420611,-0.139511,0.139511,0.56,True,True,True,Diagnostic model-sensitivity scenario; not a c...
3,36305,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.889163,0.528605,-0.360558,0.360558,0.56,True,True,True,Diagnostic model-sensitivity scenario; not a c...
4,39808,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.885159,0.425336,-0.459824,0.459824,0.56,True,True,True,Diagnostic model-sensitivity scenario; not a c...
5,41785,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.559827,0.179778,-0.380049,0.380049,0.56,False,True,False,Diagnostic model-sensitivity scenario; not a c...
6,46539,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.884053,0.543018,-0.341035,0.341035,0.56,True,True,True,Diagnostic model-sensitivity scenario; not a c...
7,46549,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.882921,0.618082,-0.264839,0.264839,0.56,True,False,False,Diagnostic model-sensitivity scenario; not a c...
8,55092,Reduce pricing rate to 8%,pricing_scenario,interest_rate,0.893485,0.564562,-0.328922,0.328922,0.56,True,False,False,Diagnostic model-sensitivity scenario; not a c...
9,57569,Increase verified annual income by 30%,affordability_scenario,total_income_pa,0.560106,0.482080,-0.078026,0.078026,0.56,True,True,True,Diagnostic model-sensitivity scenario; not a c...


## 13. Ethical and regulatory aspects

Key governance points:

- Explanations must be based on leakage-safe features and documented data lineage.
- Sensitive attributes and proxy features require review before production use.
- Counterfactuals must not be presented as guaranteed approval actions.
- SHAP and anchor-style explanations support transparency, but they do not replace model validation.
- Stakeholder communication should separate model ranking quality from operational threshold policy.

In [22]:
readiness_rows = [
    {"check": "Champion model loaded", "status": "pass", "detail": str(artifacts.model_path)},
    {"check": "Recommended threshold loaded", "status": "pass", "detail": f"{artifacts.recommended_threshold:.4f}"},
    {"check": "Classification metrics exported", "status": "pass", "detail": "08_xai_classification_metrics_at_operating_threshold.csv"},
    {"check": "Deepchecks/fallback diagnostics exported", "status": "pass", "detail": "08_deepchecks_execution_status.csv"},
    {"check": "SHAP global and local explanations exported", "status": "pass", "detail": f"{len(shap_df):,} explained rows"},
    {"check": "Anchor-style explanations exported", "status": "pass" if not anchor_rules.empty else "warning", "detail": f"{len(anchor_rules):,} rows"},
    {"check": "Counterfactual diagnostics exported", "status": "pass" if not counterfactuals.empty else "warning", "detail": f"{len(counterfactuals):,} scenario rows"},
    {"check": "Regulatory/stakeholder interpretation exported", "status": "pass", "detail": "08_xai_business_regulator_summary_model_insights.csv"},
]
readiness = build_explainability_readiness_gate(readiness_rows)
readiness.to_csv(TABLE_DIR / "08_xai_readiness_gate.csv", index=False)
readiness

,check,status,detail
0,Champion model loaded,pass,D:\Banking and Finance\Projects\canadian-retai...
1,Recommended threshold loaded,pass,0.5600
2,Classification metrics exported,pass,08_xai_classification_metrics_at_operating_thr...
3,Deepchecks/fallback diagnostics exported,pass,08_deepchecks_execution_status.csv
4,SHAP global and local explanations exported,pass,"1,500 explained rows"
5,Anchor-style explanations exported,pass,15 rows
6,Counterfactual diagnostics exported,pass,109 scenario rows
7,Regulatory/stakeholder interpretation exported,pass,08_xai_business_regulator_summary_model_insigh...


## 14. Output manifest

This manifest lists the explainability tables, figures, and optional HTML diagnostics generated by Notebook 08.

In [23]:
manifest_rows = []
for folder in [TABLE_DIR, FIGURE_DIR, HTML_DIR]:
    for path in sorted(folder.glob("08_*")):
        manifest_rows.append({"artifact_type": folder.name, "path": str(path.relative_to(PROJECT_ROOT)), "exists": path.exists()})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(TABLE_DIR / "08_xai_output_manifest.csv", index=False)
manifest

,artifact_type,path,exists
0,tables,reports\tables\08_deepchecks_execution_status.csv,True
1,tables,reports\tables\08_deepchecks_fallback_model_we...,True
2,tables,reports\tables\08_model_enhancement_recommenda...,True
3,tables,reports\tables\08_xai_anchor_style_rules.csv,True
4,tables,reports\tables\08_xai_best_counterfactual_per_...,True
5,tables,reports\tables\08_xai_business_regulator_summa...,True
6,tables,reports\tables\08_xai_classification_metrics_a...,True
7,tables,reports\tables\08_xai_confusion_matrix_interpr...,True
8,tables,reports\tables\08_xai_counterfactual_scenarios...,True
9,tables,reports\tables\08_xai_feature_alignment_audit.csv,True


## 15. Final Notebook 08 decision

Notebook 08 is complete when the readiness gate passes and the stakeholder tables explain:

- What the model gets right and wrong.
- Why recall, precision, F1, false positives, and false negatives matter.
- Which features drive global model behaviour.
- Why individual accounts receive high-risk scores.
- Which intuitive rules describe model behaviour.
- Which counterfactual scenarios change model probability.
- What governance caveats are required for ethical and regulatory communication.